# LangGraph 基礎：State、Node 與 Edge

LangGraph 可以把一個工作流程表示成圖（Graph）：

- **State**：節點之間共同傳遞的資料。
- **Node**：讀取 State、執行工作，並回傳 State 更新。
- **Edge**：決定節點的執行順序。

本例建立一個簡單的循序流程：先把數字加 3，再將結果乘以 2。

## 1. Import libraries

In [ ]:
from typing import List, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph

## 2. 定義 State

State 是整個 Graph 共用的資料結構。每個 Node 都可以讀取目前的 State，並回傳需要更新的欄位。

這裡使用 `TypedDict` 說明 State 包含：

- `value`：目前的數值。
- `history`：記錄每個節點做過的運算。

In [ ]:
class State(TypedDict):
    value: int
    history: List[str]

## 3. 定義 Nodes

Node 是接收 State 的 Python 函式。它不必回傳完整 State，只需要回傳想要更新的欄位。

本例為了清楚呈現資料變化，兩個節點都會更新 `value` 與 `history`。

In [ ]:
def add_three(state: State):
    old_value = state["value"]
    new_value = old_value + 3

    print(f"ADD THREE: {old_value} + 3 = {new_value}")

    return {
        "value": new_value,
        "history": state["history"] + [f"{old_value} + 3 = {new_value}"],
    }

In [ ]:
def multiply_by_two(state: State):
    old_value = state["value"]
    new_value = old_value * 2

    print(f"MULTIPLY BY TWO: {old_value} × 2 = {new_value}")

    return {
        "value": new_value,
        "history": state["history"] + [f"{old_value} × 2 = {new_value}"],
    }

## 4. 建立 StateGraph 並加入 Nodes

建立 `StateGraph` 時要指定 State 的資料結構，再使用 `add_node()` 為每個函式命名。

In [ ]:
workflow = StateGraph(State)

workflow.add_node("add_three", add_three)
workflow.add_node("multiply_by_two", multiply_by_two)

## 5. 使用 Edges 連接執行順序

Edge 表示節點之間的方向：

1. `START → add_three`：從加法節點開始。
2. `add_three → multiply_by_two`：完成加法後進行乘法。
3. `multiply_by_two → END`：完成乘法後結束。

In [ ]:
workflow.add_edge(START, "add_three")
workflow.add_edge("add_three", "multiply_by_two")
workflow.add_edge("multiply_by_two", END)

## 6. Compile Graph

`compile()` 會把工作流程轉換成可以執行的 Graph。

In [ ]:
graph = workflow.compile()

## 7. 顯示 Graph

圖中可以直接看到 START、Nodes、Edges 與 END 的關係。

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 8. 執行 Graph

輸入的初始 State 為 `value = 5`。執行過程是 `(5 + 3) × 2`，所以最後的 `value` 應為 16。

In [ ]:
initial_state = {
    "value": 5,
    "history": [],
}

result = graph.invoke(initial_state)

In [ ]:
print("Final value:", result["value"])
print("History:")
for step in result["history"]:
    print("-", step)

## State 如何在 Nodes 之間傳遞

```text
Initial State
{value: 5, history: []}
        │
        ▼
add_three
{value: 8, history: ["5 + 3 = 8"]}
        │
        ▼
multiply_by_two
{value: 16, history: ["5 + 3 = 8", "8 × 2 = 16"]}
        │
        ▼
END
```

## 重點整理

| 概念 | 用途 | 本例 |
|---|---|---|
| State | 儲存並傳遞工作流程資料 | `value`、`history` |
| Node | 讀取 State 並回傳更新 | `add_three`、`multiply_by_two` |
| Edge | 定義節點的執行方向 | `add_three → multiply_by_two` |
| START | Graph 的起點 | 前往 `add_three` |
| END | Graph 的終點 | 在乘法完成後結束 |
| compile | 將流程轉成可執行 Graph | `workflow.compile()` |
| invoke | 傳入初始 State 並執行 | `graph.invoke(initial_state)` |